# YaRN

In [1]:
import torch
import torch.nn
import torch.nn.functional as F
from dataclasses import dataclass
import math

## RoPE

In [2]:
def _apply_rotary_emb(
    x: torch.Tensor,
    cos: torch.Tensor,
    sin: torch.Tensor,
) -> torch.Tensor:
    cos = cos.unsqueeze(-2).to(x.dtype)
    sin = sin.unsqueeze(-2).to(x.dtype)
    x1, x2 = torch.chunk(x, 2, dim=-1)
    o1 = x1 * cos - x2 * sin
    o2 = x2 * cos + x1 * sin
    return torch.cat((o1, o2), dim=-1)

In [3]:
class RoPE(torch.nn.Module):
    def __init__(
        self,
        head_dim: int,
        base: int,
    ) -> None:
        super().__init__()
        self.head_dim = head_dim
        self.base = base
        self.cos, self.sin = self._compute_cos_sin(num_tokens)

    def _compute_concentration_and_inv_freq(self) -> torch.Tensor:
        freq = self.base ** (torch.arange(0, self.head_dim, 2,)/ self.head_dim)
        inv_freq = 1.0 / freq
        return inv_freq

    def _compute_cos_sin(self, num_tokens: int):
        inv_freq = self._compute_concentration_and_inv_freq()
        t = torch.arange(num_tokens)
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        cos = freqs.cos() 
        sin = freqs.sin() 
        return cos, sin

    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        num_tokens = query.shape[0]
            
        query_shape = query.shape
        query = query.view(num_tokens, -1, self.head_dim)
        query = _apply_rotary_emb(query, cos, sin)
        query = query.reshape(query_shape)

        key_shape = key.shape
        key = key.view(num_tokens, -1, self.head_dim)
        key = _apply_rotary_emb(key, cos, sin)
        key = key.reshape(key_shape)
        return query, key